In [1]:
!pip install -q transformers datasets torchaudio librosa scikit-learn accelerate torchcodec


In [2]:
import os
import random
import torch
import torchaudio
import numpy as np

from torch.utils.data import Dataset
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    DataCollatorWithPadding,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import KFold

In [3]:
# -------------------------
# Configuración general
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATASET_PATH = "/content/dataset"
SAMPLE_RATE = 16000
NUM_LABELS = 3
SEED = 42
K_FOLDS = 3

random.seed(SEED)
torch.manual_seed(SEED)


In [4]:
# -------------------------
# Mapeo de emociones
# -------------------------
emotion2id = {
    "alegria": 0,
    "neutral": 1,
    "tristeza": 2,
    "miedo": 2,
    "sorpresa": 2,
    "enojo": 2,
    "asco": 2
}

id2emotion = {
    0: "positivo",
    1: "neutral",
    2: "negativo"
}

In [5]:
class EmotionDataset(Dataset):

    def __init__(self, files, data_dir, processor):
        self.files = files
        self.data_dir = data_dir
        self.processor = processor

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        filename = self.files[idx]
        path = os.path.join(self.data_dir, filename)

        # Etiqueta desde el nombre
        emotion = filename.split("_")[-1].replace(".wav", "").lower()
        label = emotion2id[emotion]

        # Cargar audio
        speech, sr = torchaudio.load(path)

        if sr != SAMPLE_RATE:
            speech = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(speech)

        # Concatenar canales
        speech_concat = torch.cat([speech[0], speech[1]], dim=0)

        inputs = self.processor(
            speech_concat.numpy(),
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=True
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels": torch.tensor(label)
        }


In [6]:
# -------------------------
# Modelo y procesador
# -------------------------
MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-spanish"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=processor,
    padding=True,
    return_tensors="pt"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [7]:
# -------------------------
# Métricas
# -------------------------
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="weighted",
        zero_division=0
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1-score": f1
    }

In [8]:
# -------------------------
# Argumentos de entrenamiento
# -------------------------
training_args = TrainingArguments(

    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=8,
    warmup_steps=10,
    logging_dir="./logs",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1-score",
    fp16=False
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
# -------------------------
# Obtener lista de audios
# -------------------------
all_files = [f for f in os.listdir(DATASET_PATH) if f.lower().endswith(".wav")]

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

results_all = []

In [10]:
# -------------------------
# Cross Validation
# -------------------------
for fold, (train_idx, test_idx) in enumerate(kf.split(all_files)):

    print(f"\n==========================")
    print(f"FOLD {fold+1}/{K_FOLDS}")
    print(f"==========================")

    train_files = [all_files[i] for i in train_idx]
    test_files = [all_files[i] for i in test_idx]

    print("Train:", len(train_files))
    print("Test:", len(test_files))

    train_dataset = EmotionDataset(train_files, DATASET_PATH, processor)
    test_dataset = EmotionDataset(test_files, DATASET_PATH, processor)

    # Reinicializar modelo en cada fold
    model = Wav2Vec2ForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS
    ).to(DEVICE)

    # Congelar extractor
    for param in model.wav2vec2.feature_extractor.parameters():
        param.requires_grad = False

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        processing_class=processor,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    results = trainer.evaluate()

    results_all.append(results)


FOLD 1/3
Train: 48
Test: 24


pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-spanish
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
projector.bias    | MISSING    | 
classifier.bias   | MISSING    | 
projector.weight  | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 72.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 27.81 MiB is free. Including non-PyTorch memory, this process has 14.53 GiB memory in use. Of the allocated memory 14.22 GiB is allocated by PyTorch, and 181.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# -------------------------
# Promedio, media y varianza
# -------------------------
print("\n==========================")
print("RESULTADOS CROSS VALIDATION")
print("==========================")

acc_list = [r["eval_accuracy"] for r in results_all]
prec_list = [r["eval_precision"] for r in results_all]
rec_list = [r["eval_recall"] for r in results_all]
f1_list = [r["eval_f1-score"] for r in results_all]

# Media
acc_mean = np.mean(acc_list)
prec_mean = np.mean(prec_list)
rec_mean = np.mean(rec_list)
f1_mean = np.mean(f1_list)

# Varianza
acc_var = np.var(acc_list)
prec_var = np.var(prec_list)
rec_var = np.var(rec_list)
f1_var = np.var(f1_list)

# Desviación estándar
acc_std = np.std(acc_list)
prec_std = np.std(prec_list)
rec_std = np.std(rec_list)
f1_std = np.std(f1_list)

print(f"Accuracy -> media: {acc_mean:.4f} | varianza: {acc_var:.6f} | std: {acc_std:.4f}")
print(f"Precision -> media: {prec_mean:.4f} | varianza: {prec_var:.6f} | std: {prec_std:.4f}")
print(f"Recall -> media: {rec_mean:.4f} | varianza: {rec_var:.6f} | std: {rec_std:.4f}")
print(f"F1-score -> media: {f1_mean:.4f} | varianza: {f1_var:.6f} | std: {f1_std:.4f}")